# 📟 Multimeter Screen Detection & OCR Pipeline v0.02 (Two-Stage)

**Goal:** Run a two-stage detection pipeline to first isolate the multimeter body, then isolate the LCD screen inside it, before running OCR.

**Pipeline:**
1. **Stage 1 (Grounding DINO):** Detect 'multimeter body' in full image.
2. **Stage 2 (Grounding DINO):** Detect 'LCD screen' in the cropped meter image.
3. **Preprocessing:** Resize and enhance contrast of the LCD crop for better OCR.
4. **OCR (EasyOCR):** Extract digits from the LCD crop.
5. **Export:** Save all crops, annotated images, and `results.csv`.


## 1️⃣ Imports & Setup

In [48]:
import os
import csv
import logging
from pathlib import Path
from datetime import datetime

import cv2
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

# Fix Windows encoding for EasyOCR progress bars
os.environ['PYTHONIOENCODING'] = 'utf-8'

# --- Logging setup ---
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)
log_file = LOG_DIR / f"run_v2_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info(f"Log file: {log_file}")

# --- Device ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-05-14 17:04:47,287 [INFO] Log file: logs\run_v2_20260514_170447.log
2026-05-14 17:04:47,288 [INFO] Using device: cuda
2026-05-14 17:04:47,290 [INFO] GPU: NVIDIA GeForce GTX 1650
2026-05-14 17:04:47,291 [INFO] VRAM: 4.3 GB


## 2️⃣ Configuration

In [49]:
# === PATHS ===
BASE_DIR = Path(".")
DATA_FOLDERS = [
    BASE_DIR / "Data" / "N to E",
    BASE_DIR / "Data" / "P to E",
    BASE_DIR / "Data" / "P to N",
]
OUTPUT_DIR = BASE_DIR / "output"

DIRS = {
    "s1_ann": OUTPUT_DIR / "stage1_full_annotated",
    "s1_crop": OUTPUT_DIR / "stage1_meter_crops",
    "s2_ann": OUTPUT_DIR / "stage2_meter_annotated",
    "s2_raw": OUTPUT_DIR / "stage2_lcd_crops_raw",
    "s2_proc": OUTPUT_DIR / "stage2_lcd_crops_processed",
    "final_ann": OUTPUT_DIR / "final_full_annotated",
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "results.csv"

# === MODEL ===
MODEL_ID = "IDEA-Research/grounding-dino-tiny"

# === PROMPTS & THRESHOLDS ===
METER_PROMPT = "Kazam EV charger wall box . handheld digital multimeter ."
LCD_PROMPT = "small rectangular LCD display screen on multimeter, grey green digital display, only seven segment digits, numeric reading screen, not rotary dial, not labels, not buttons, not ports."

STAGE1_BOX_THRESHOLD = 0.20
STAGE1_TEXT_THRESHOLD = 0.20
STAGE1_MIN_AREA_RATIO = 0.01
STAGE1_MAX_AREA_RATIO = 0.40  # Reject large charger boxes

STAGE2_BOX_THRESHOLD = 0.15
STAGE2_TEXT_THRESHOLD = 0.15
STAGE2_MIN_AREA_RATIO = 0.04  # LCD relative to meter crop
STAGE2_MAX_AREA_RATIO = 0.50  # LCD shouldn't be the entire meter

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

logger.info("Configuration loaded.")

2026-05-14 17:04:47,345 [INFO] Configuration loaded.


## 3️⃣ Collect Image Paths

In [50]:
image_list = []

for folder in DATA_FOLDERS:
    if not folder.exists():
        continue
    for f in sorted(folder.iterdir()):
        if f.suffix.lower() in IMG_EXTS:
            image_list.append((f, folder.name))

logger.info(f"Total images found: {len(image_list)}")

2026-05-14 17:04:47,425 [INFO] Total images found: 58


## 4️⃣ Load Grounding DINO Model

In [51]:
logger.info(f"Loading Grounding DINO from {MODEL_ID} ...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

logger.info("Grounding DINO loaded successfully!")

2026-05-14 17:04:47,487 [INFO] Loading Grounding DINO from IDEA-Research/grounding-dino-tiny ...
2026-05-14 17:04:52,746 [INFO] Grounding DINO loaded successfully!


## 5️⃣ Helper Functions

In [52]:
def detect_objects(image_input, prompt, box_thresh, text_thresh):
    """Run Grounding DINO on an image path or PIL Image."""
    if isinstance(image_input, Path) or isinstance(image_input, str):
        image = Image.open(image_input).convert("RGB")
    else:
        image = image_input

    w, h = image.size
    img_area = w * h

    inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs, inputs["input_ids"], threshold=box_thresh, text_threshold=text_thresh, target_sizes=[(h, w)]
    )[0]

    detections = []
    for bbox, score, label in zip(results["boxes"], results["scores"], results.get("text_labels", [""]*len(results["scores"]))):
        box = bbox.cpu().numpy().astype(int).tolist()
        x1, y1, x2, y2 = box
        area_ratio = ((x2 - x1) * (y2 - y1)) / img_area
        detections.append({"bbox": box, "score": float(score), "label": label, "area_ratio": area_ratio})
    return detections

def filter_detections(detections, min_area, max_area, require_horizontal=False):
    """Filter by area limits, and optionally enforce width > height for LCDs."""
    filtered = []
    for det in detections:
        x1, y1, x2, y2 = det["bbox"]
        if min_area <= det["area_ratio"] <= max_area:
            if require_horizontal:
                if (x2 - x1) > (y2 - y1) * 0.9: # slightly relaxed horizontal check
                    filtered.append(det)
            else:
                filtered.append(det)
    filtered.sort(key=lambda d: d["score"], reverse=True)
    return filtered

def preprocess_lcd_crop(crop_img):
    """Enhance crop for 7-segment OCR reading using adaptive thresholding."""
    h, w = crop_img.shape[:2]
    # 1. Resize to make digits larger for OCR engine
    resized = cv2.resize(crop_img, (w*3, h*3), interpolation=cv2.INTER_CUBIC)
    
    # 2. Grayscale
    gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    
    # 3. Contrast enhancement (CLAHE)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    contrast = clahe.apply(gray)
    
    # 4. Blur slightly to remove tiny noise specks
    blurred = cv2.GaussianBlur(contrast, (5, 5), 0)
    
    # 5. Adaptive Thresholding (Creates a pure black and white image)
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
    
    return thresh


## 6️⃣ Stage 1 & 2 Pipeline Loop

In [53]:
results_list = []

for img_path, folder_name in tqdm(image_list, desc="Running 2-Stage DINO"):
    safe_name = img_path.stem.replace(" ", "_").replace("(", "").replace(")", "")
    row = {
        "image_path": str(img_path),
        "connection_type_folder": folder_name,
        "meter_detected": False, "meter_bbox_global": "", "stage1_meter_crop_path": "",
        "lcd_detected": False, "lcd_bbox_global": "", "lcd_crop_path": "",
        "lcd_crop_raw_path": "", "lcd_crop_processed_path": "",
        "easyocr_debug_text": "",
        "seven_segment_display_string": "", "seven_segment_numeric_value": "",
        "seven_segment_confidence": "", "seven_segment_status": "", "seven_segment_failure_reason": "",
        "final_value": "", "final_status": "", "failure_reason": ""
    }
    
    img_cv2 = cv2.imread(str(img_path))
    if img_cv2 is None:
        row["final_status"] = "error"
        row["failure_reason"] = "could not read image"
        results_list.append(row)
        continue
        
    # =======================================
    # STAGE 1: Detect Meter in Full Image
    # =======================================
    try:
        raw_s1 = detect_objects(img_path, METER_PROMPT, STAGE1_BOX_THRESHOLD, STAGE1_TEXT_THRESHOLD)
        
        # Filter out the charger class (Plan A logic)
        meter_only_s1 = []
        for det in raw_s1:
            lbl = det["label"].lower()
            if "multimeter" in lbl or "handheld" in lbl or lbl == "":
                meter_only_s1.append(det)
                
        filtered_s1 = filter_detections(meter_only_s1, STAGE1_MIN_AREA_RATIO, STAGE1_MAX_AREA_RATIO)
    except Exception as e:
        row["final_status"] = "error"
        row["failure_reason"] = f"stage1_error: {e}"
        results_list.append(row)
        continue
        
    if not filtered_s1:
        row["final_status"] = "meter_not_detected"
        row["failure_reason"] = "no meter found in stage 1"
        results_list.append(row)
        continue
        
    best_meter = filtered_s1[0]
    mx1, my1, mx2, my2 = best_meter["bbox"]
    row["meter_detected"] = True
    row["stage1_meter_confidence"] = f"{best_meter['score']:.4f}"
    row["meter_bbox_global"] = str([mx1, my1, mx2, my2])
    
    # Save S1 Ann
    s1_ann_img = img_cv2.copy()
    cv2.rectangle(s1_ann_img, (mx1, my1), (mx2, my2), (0, 255, 0), 3)
    cv2.imwrite(str(DIRS["s1_ann"] / f"{folder_name}_{safe_name}_s1.jpg"), s1_ann_img)
    
    # Save S1 Crop
    meter_crop = img_cv2[my1:my2, mx1:mx2]
    if meter_crop.size == 0:
        row["final_status"] = "error"
        row["failure_reason"] = "empty meter crop"
        results_list.append(row)
        continue
    s1_crop_path = DIRS["s1_crop"] / f"{folder_name}_{safe_name}_meter.jpg"
    cv2.imwrite(str(s1_crop_path), meter_crop)
    row["stage1_meter_crop_path"] = str(s1_crop_path)
    
    # =======================================
    # STAGE 2: Detect LCD in Meter Crop
    # =======================================
    try:
        meter_pil = Image.fromarray(cv2.cvtColor(meter_crop, cv2.COLOR_BGR2RGB))
        raw_s2 = detect_objects(meter_pil, LCD_PROMPT, STAGE2_BOX_THRESHOLD, STAGE2_TEXT_THRESHOLD)
        filtered_s2 = filter_detections(raw_s2, STAGE2_MIN_AREA_RATIO, STAGE2_MAX_AREA_RATIO, require_horizontal=True)
        if not filtered_s2 and raw_s2: # fallback if horizontal check fails
            filtered_s2 = filter_detections(raw_s2, STAGE2_MIN_AREA_RATIO, STAGE2_MAX_AREA_RATIO, require_horizontal=False)
    except Exception as e:
        row["final_status"] = "error"
        row["failure_reason"] = f"stage2_error: {e}"
        results_list.append(row)
        continue
        
    if not filtered_s2:
        row["final_status"] = "lcd_not_detected"
        row["failure_reason"] = "no lcd found in stage 2"
        results_list.append(row)
        continue
        
    best_lcd = filtered_s2[0]
    lx1, ly1, lx2, ly2 = best_lcd["bbox"]
    # Convert to global coordinates
    gx1, gy1, gx2, gy2 = lx1 + mx1, ly1 + my1, lx2 + mx1, ly2 + my1
    
    row["lcd_detected"] = True
    row["stage2_lcd_confidence"] = f"{best_lcd['score']:.4f}"
    row["stage2_lcd_bbox_local"] = str([lx1, ly1, lx2, ly2])
    row["lcd_bbox_global"] = str([gx1, gy1, gx2, gy2])
    
    # Save S2 Ann (on meter crop)
    s2_ann_img = meter_crop.copy()
    cv2.rectangle(s2_ann_img, (lx1, ly1), (lx2, ly2), (0, 165, 255), 2) # Orange for local LCD
    cv2.imwrite(str(DIRS["s2_ann"] / f"{folder_name}_{safe_name}_lcd_ann.jpg"), s2_ann_img)
    
    # Save Final Full Ann (Meter=Green, LCD=Red)
    final_ann_img = img_cv2.copy()
    cv2.rectangle(final_ann_img, (mx1, my1), (mx2, my2), (0, 255, 0), 3) # Meter
    cv2.rectangle(final_ann_img, (gx1, gy1), (gx2, gy2), (0, 0, 255), 2) # LCD
    cv2.imwrite(str(DIRS["final_ann"] / f"{folder_name}_{safe_name}_final.jpg"), final_ann_img)
    
    # Save LCD Crops
    lcd_crop = meter_crop[ly1:ly2, lx1:lx2]
    if lcd_crop.size == 0:
        row["final_status"] = "error"
        row["failure_reason"] = "empty lcd crop"
        results_list.append(row)
        continue
        
    raw_lcd_path = DIRS["s2_raw"] / f"{folder_name}_{safe_name}_lcd_raw.jpg"
    cv2.imwrite(str(raw_lcd_path), lcd_crop)
    row["lcd_crop_raw_path"] = str(raw_lcd_path)
    row["lcd_crop_path"] = str(raw_lcd_path)
    
    proc_lcd = preprocess_lcd_crop(lcd_crop)
    proc_lcd_path = DIRS["s2_proc"] / f"{folder_name}_{safe_name}_lcd_proc.jpg"
    cv2.imwrite(str(proc_lcd_path), proc_lcd)
    row["lcd_crop_processed_path"] = str(proc_lcd_path)
    
    row["final_status"] = "ready_for_ocr"
    results_list.append(row)

logger.info("Stage 1 & 2 complete.")

Running 2-Stage DINO: 100%|██████████| 58/58 [02:02<00:00,  2.11s/it]
2026-05-14 17:06:55,274 [INFO] Stage 1 & 2 complete.


## 7️⃣ Run EasyOCR on Preprocessed LCD Crops

In [54]:
import easyocr
import cv2
from src.ocr.seven_segment_reader import read_lcd_value, SevenSegmentConfig

ocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == "cuda"))
ss_config = SevenSegmentConfig()

ss_success = 0
ss_fail = 0

for row in tqdm(results_list, desc="Running OCR & 7-Segment Reader"):
    if row["final_status"] != "ready_for_ocr":
        continue
        
    raw_path = row.get("lcd_crop_raw_path", "")
    proc_path = row.get("lcd_crop_processed_path", "")
    
    # 1. EasyOCR (Debug only)
    if proc_path and Path(proc_path).exists():
        try:
            result = ocr_reader.readtext(proc_path, allowlist='0123456789.-', text_threshold=0.5)
            texts = [t[1] for t in result]
            if texts:
                row["easyocr_debug_text"] = " | ".join(texts)
        except Exception:
            pass
            
    # 2. Seven Segment Reader
    if raw_path and Path(raw_path).exists():
        try:
            lcd_img = cv2.imread(raw_path)
            safe_name = Path(raw_path).stem.replace("_lcd_raw", "")
            ss_result = read_lcd_value(lcd_img, safe_name=safe_name, config=ss_config)
            
            row["seven_segment_display_string"] = ss_result["display_string"]
            row["seven_segment_numeric_value"] = ss_result["numeric_value"]
            row["seven_segment_confidence"] = f"{ss_result['confidence']:.4f}"
            row["seven_segment_status"] = ss_result["status"]
            row["seven_segment_failure_reason"] = ss_result["failure_reason"]
            
            if ss_result["status"] == "success":
                row["final_status"] = "success"
                row["final_value"] = ss_result["numeric_value"]
                ss_success += 1
                logger.info(f"[7-Seg] {Path(row['image_path']).name} -> {ss_result['display_string']}")
            else:
                row["final_status"] = ss_result["status"]
                row["failure_reason"] = ss_result["failure_reason"]
                ss_fail += 1
        except Exception as e:
            row["final_status"] = "digit_decode_failed"
            row["failure_reason"] = f"exception: {e}"
            ss_fail += 1

logger.info(f"7-Segment Reader complete: {ss_success} success, {ss_fail} failed")

Running OCR & 7-Segment Reader: 100%|██████████| 58/58 [00:05<00:00, 10.25it/s]
2026-05-14 17:07:03,747 [INFO] 7-Segment Reader complete: 0 success, 58 failed


## 8️⃣ Save Results CSV

In [55]:
csv_columns = [
    "image_path", "connection_type_folder", 
    "meter_detected", "meter_bbox_global",
    "lcd_detected", "lcd_bbox_global", "lcd_crop_path",
    "easyocr_debug_text",
    "seven_segment_display_string", "seven_segment_numeric_value", 
    "seven_segment_confidence", "seven_segment_status", "seven_segment_failure_reason",
    "final_value", "final_status", "failure_reason"
]

with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
    # filter row keys to only those in csv_columns
    filtered_results = []
    for r in results_list:
        filtered_results.append({k: r.get(k, "") for k in csv_columns})
        
    writer = csv.DictWriter(f, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(filtered_results)

logger.info(f"Results saved to {CSV_PATH}")
print(f"\n\u2705 CSV saved: {CSV_PATH}")

2026-05-14 17:07:03,807 [INFO] Results saved to output\results.csv



✅ CSV saved: output\results.csv


## 9️⃣ Summary

In [56]:
total = len(image_list)
s1_fail = sum(1 for r in results_list if r["final_status"] == "meter_not_detected")
s2_fail = sum(1 for r in results_list if r["final_status"] == "lcd_not_detected")
ocr_fail = sum(1 for r in results_list if r["final_status"] == "ocr_failed")
success = sum(1 for r in results_list if r["final_status"] == "success")

print("=" * 60)
print("  2-STAGE PIPELINE SUMMARY (v0.02)")
print("=" * 60)
print(f"  Total images processed : {total}")
print(f"  Meter Detection Failed : {s1_fail}")
print(f"  LCD Detection Failed   : {s2_fail}")
print(f"  OCR Extraction Failed  : {ocr_fail}")
print(f"  Success (Text Found)   : {success}")
print("=" * 60)

if success > 0:
    print("\n  Successful Extractions:")
    for r in results_list:
        if r["final_status"] == "success":
            print(f"    [{r['folder']}] {Path(r['image_path']).name}: {r['ocr_text']}")

logger.info("Pipeline v0.02 complete.")

2026-05-14 17:07:03,878 [INFO] Pipeline v0.02 complete.


  2-STAGE PIPELINE SUMMARY (v0.02)
  Total images processed : 58
  Meter Detection Failed : 0
  LCD Detection Failed   : 0
  OCR Extraction Failed  : 0
  Success (Text Found)   : 0


## 🔟 Evaluation Mode

In [58]:
import pandas as pd
import cv2
from src.ocr.seven_segment_reader import read_lcd_value, SevenSegmentConfig

# Place an evaluation.csv with columns: lcd_crop_path, true_value
EVAL_CSV = BASE_DIR / "evaluation.csv"

if EVAL_CSV.exists():
    df = pd.read_csv(EVAL_CSV)
    ss_config = SevenSegmentConfig()
    
    exact_match = 0
    numeric_match = 0
    total = len(df)
    failed = 0
    ambiguous = 0
    
    for idx, row in df.iterrows():
        img_path = row['lcd_crop_path']
        true_val = str(row['true_value'])
        
        if not Path(img_path).exists():
            continue
            
        img = cv2.imread(img_path)
        res = read_lcd_value(img, safe_name=f"eval_{idx}", config=ss_config)
        
        pred_str = res['display_string']
        status = res['status']
        
        if status == 'success':
            if pred_str == true_val:
                exact_match += 1
            try:
                if float(pred_str) == float(true_val):
                    numeric_match += 1
            except:
                pass
        elif status == 'digit_decode_failed':
            ambiguous += 1
        else:
            failed += 1
            
    print(f"Evaluation on {total} images:")
    print(f"Exact String Accuracy: {exact_match/total*100:.1f}%")
    print(f"Numeric Accuracy: {numeric_match/total*100:.1f}%")
    print(f"Failed: {failed}")
    print(f"Ambiguous: {ambiguous}")
else:
    print("No evaluation.csv found. Create one to run evaluation metrics.")

EmptyDataError: No columns to parse from file